In [2]:
from __future__ import annotations

import json
import tempfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms


# Блок 1. Конфигурация (все основные параметры в одном месте)
CFG = {
    "seed": 42,
    "data": {
        "dataset": "CIFAR10",
        "val_ratio": 0.2,
        "batch_size": 128,
        "num_workers": 0,
        # Быстрый режим: берем подвыборку, чтобы на CPU не ждать слишком долго
        "fast_mode": True,
        "max_train_samples": 12000,
        "max_test_samples": 5000,
        # Нормализация CIFAR10 (стандартные mean/std)
        "normalize_mean": (0.4914, 0.4822, 0.4465),
        "normalize_std": (0.2470, 0.2435, 0.2616),
    },
    "train": {
        "max_epochs_base": 20,  # для E1–E3
        "max_epochs_best": 50,  # для E4 (с EarlyStopping)
        "early_stopping_patience": 4,
        "early_stopping_min_delta": 0.0005,
        # Логи по эпохам: чтобы вывод не был огромным
        "verbose": False,
        "print_every": 5,
    },
}

# Подавляем предупреждение torchvision + NumPy (не влияет на обучение)
warnings.filterwarnings(
    "ignore",
    message=r"dtype\(\): align should be passed.*",
)

SEED = int(CFG["seed"])


# Блок 2. Воспроизводимость и устройство
def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# Блок 3. Папки проекта (данные и артефакты)
def resolve_hw_dir() -> Path:
    # Чтобы ноутбук работал независимо от того, где стартовал Jupyter
    candidates = [
        Path.cwd(),
        Path.cwd() / "HW08-09",
        Path.cwd() / "homeworks" / "HW08-09",
        Path.cwd() / "makaroni_po_flotski" / "homeworks" / "HW08-09",
    ]
    for c in candidates:
        if (c / "HW08-09.ipynb").exists() or (c / "practic").exists():
            return c
    return Path.cwd()


def ensure_writable_dir(path: Path, purpose: str) -> Path:
    try:
        path.mkdir(parents=True, exist_ok=True)
        probe = path / ".write_probe"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink(missing_ok=True)
        return path
    except Exception as e:
        alt = Path(tempfile.gettempdir()) / "hw08_09" / purpose
        alt.mkdir(parents=True, exist_ok=True)
        print(f"WARNING: can't write to {path} ({purpose}): {e}. Using {alt}")
        return alt


hw_dir = resolve_hw_dir()
data_root = ensure_writable_dir(hw_dir / "data", purpose="data")
artifacts_dir = ensure_writable_dir(hw_dir / "artifacts", purpose="artifacts")
fig_dir = artifacts_dir / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)


# Блок 4. Данные (CIFAR10) + DataLoader
DATA_CFG = CFG["data"]

transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(DATA_CFG["normalize_mean"], DATA_CFG["normalize_std"]),
    ]
)

dataset_name = DATA_CFG["dataset"]
num_classes = 10

train_full = datasets.CIFAR10(
    root=str(data_root), train=True, download=True, transform=transform
)
test_ds = datasets.CIFAR10(
    root=str(data_root), train=False, download=True, transform=transform
)

if DATA_CFG["fast_mode"]:
    rng = np.random.RandomState(SEED)
    train_idx = rng.choice(
        len(train_full),
        size=min(int(DATA_CFG["max_train_samples"]), len(train_full)),
        replace=False,
    )
    test_idx = rng.choice(
        len(test_ds),
        size=min(int(DATA_CFG["max_test_samples"]), len(test_ds)),
        replace=False,
    )
    train_full = Subset(train_full, train_idx)
    test_ds = Subset(test_ds, test_idx)

val_ratio = float(DATA_CFG["val_ratio"])
n_train = int((1 - val_ratio) * len(train_full))
n_val = len(train_full) - n_train

train_ds, val_ds = random_split(
    train_full,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)

batch_size = int(DATA_CFG["batch_size"])
num_workers = int(DATA_CFG["num_workers"])

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

x_batch, y_batch = next(iter(train_loader))
input_dim = int(np.prod(x_batch.shape[1:]))

print("Batch:", x_batch.shape, y_batch.shape)
print("Dataset:", dataset_name, "| input_dim:", input_dim, "| num_classes:", num_classes)


# Блок 5. Модель (MLP)
class MLP(nn.Module):
    def __init__(self, hidden_sizes=(256, 256), dropout_p=0.0, use_batchnorm=False):
        super().__init__()

        layers: list[nn.Module] = [nn.Flatten()]
        in_dim = input_dim

        for h in hidden_sizes:
            layers.append(nn.Linear(in_dim, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout_p > 0:
                layers.append(nn.Dropout(dropout_p))
            in_dim = h

        layers.append(nn.Linear(in_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# Блок 6. Метрики и цикл обучения (train/eval)
def accuracy_from_logits(logits, targets) -> float:
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_acc, total_count = 0.0, 0.0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        bs = y.size(0)
        total_loss += loss.item() * bs
        total_acc += accuracy_from_logits(logits, y) * bs
        total_count += bs

    return total_loss / total_count, total_acc / total_count


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_acc, total_count = 0.0, 0.0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        bs = y.size(0)
        total_loss += loss.item() * bs
        total_acc += accuracy_from_logits(logits, y) * bs
        total_count += bs

    return total_loss / total_count, total_acc / total_count


# Блок 7. Один эксперимент (с возможным EarlyStopping)
def run_experiment(
    experiment_id: str,
    hidden_sizes=(256, 256),
    dropout_p=0.0,
    use_batchnorm=False,
    optimizer_name="Adam",
    lr=1e-3,
    momentum=0.0,
    weight_decay=0.0,
    max_epochs=20,
    early_stopping_patience=None,
    early_stopping_min_delta=0.0,
):
    model = MLP(hidden_sizes, dropout_p, use_batchnorm).to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        momentum_to_log = 0.0
    elif optimizer_name == "SGD":
        optimizer = optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=momentum,
            weight_decay=weight_decay,
        )
        momentum_to_log = float(momentum)
    else:
        raise ValueError("optimizer_name must be 'Adam' or 'SGD'")

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    best_val_acc = -1.0
    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(1, max_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if CFG["train"].get("verbose", False):
            pe = int(CFG["train"].get("print_every", 5))
            if epoch == 1 or epoch == max_epochs or (pe > 0 and epoch % pe == 0):
                print(f"[{experiment_id}] Эпоха {epoch}: val_acc={val_acc:.4f}")

        improved = val_acc > best_val_acc + early_stopping_min_delta
        if improved:
            best_val_acc = float(val_acc)
            best_val_loss = float(val_loss)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if early_stopping_patience is not None and patience_counter >= early_stopping_patience:
            print(f"[{experiment_id}] EarlyStopping: остановка (patience={early_stopping_patience})")
            break

    epochs_trained = len(history["val_acc"])
    print(f"[{experiment_id}] Итог: best_val_acc={best_val_acc:.4f} | epochs={epochs_trained}")

    model_summary = f"hidden={hidden_sizes} | act=ReLU | dropout={dropout_p} | bn={use_batchnorm}"

    return {
        "experiment_id": experiment_id,
        "history": history,
        "epochs_trained": epochs_trained,
        "best_val_accuracy": best_val_acc,
        "best_val_loss": best_val_loss,
        "best_state_dict": best_state,
        "model_summary": model_summary,
        "config": {
            "hidden_sizes": hidden_sizes,
            "dropout_p": dropout_p,
            "use_batchnorm": use_batchnorm,
            "optimizer": optimizer_name,
            "lr": lr,
            "momentum": momentum_to_log,
            "weight_decay": weight_decay,
            "dataset": dataset_name,
            "seed": SEED,
            "fast_mode": bool(DATA_CFG["fast_mode"]),
        },
    }


# Блок 8. Эксперименты
runs = []

print("\nЧасть A (S08): регуляризация")

E1 = run_experiment(
    "E1",
    hidden_sizes=(256, 256),
    dropout_p=0.0,
    use_batchnorm=False,
    max_epochs=int(CFG["train"]["max_epochs_base"]),
)
runs.append(E1)

E2 = run_experiment(
    "E2",
    hidden_sizes=(256, 256),
    dropout_p=0.3,
    use_batchnorm=False,
    max_epochs=int(CFG["train"]["max_epochs_base"]),
)
runs.append(E2)

E3 = run_experiment(
    "E3",
    hidden_sizes=(256, 256),
    dropout_p=0.0,
    use_batchnorm=True,
    max_epochs=int(CFG["train"]["max_epochs_base"]),
)
runs.append(E3)

best_reg = max([E2, E3], key=lambda r: r["best_val_accuracy"])
print("\nЛучшая регуляризация:", best_reg["experiment_id"], "val_acc=", best_reg["best_val_accuracy"])

cfg = best_reg["config"]
E4 = run_experiment(
    "E4",
    hidden_sizes=cfg["hidden_sizes"],
    dropout_p=cfg["dropout_p"],
    use_batchnorm=cfg["use_batchnorm"],
    optimizer_name=cfg["optimizer"],
    lr=cfg["lr"],
    max_epochs=int(CFG["train"]["max_epochs_best"]),
    early_stopping_patience=int(CFG["train"]["early_stopping_patience"]),
    early_stopping_min_delta=float(CFG["train"]["early_stopping_min_delta"]),
)
runs.append(E4)

best_overall = E4
best_cfg = best_overall["config"]

print("\nЧасть B (S09): LR, оптимизаторы, weight decay")

O1 = run_experiment(
    "O1",
    hidden_sizes=best_cfg["hidden_sizes"],
    dropout_p=best_cfg["dropout_p"],
    use_batchnorm=best_cfg["use_batchnorm"],
    optimizer_name="Adam",
    lr=1e-1,
    max_epochs=6,
)
runs.append(O1)

O2 = run_experiment(
    "O2",
    hidden_sizes=best_cfg["hidden_sizes"],
    dropout_p=best_cfg["dropout_p"],
    use_batchnorm=best_cfg["use_batchnorm"],
    optimizer_name="Adam",
    lr=1e-5,
    max_epochs=6,
)
runs.append(O2)

O3 = run_experiment(
    "O3",
    hidden_sizes=best_cfg["hidden_sizes"],
    dropout_p=best_cfg["dropout_p"],
    use_batchnorm=best_cfg["use_batchnorm"],
    optimizer_name="SGD",
    lr=1e-2,
    momentum=0.9,
    weight_decay=1e-4,
    max_epochs=15,
)
runs.append(O3)


# Блок 9. Сохранение артефактов
rows = []
for r in runs:
    cfg = r["config"]
    rows.append(
        {
            "experiment_id": r["experiment_id"],
            "dataset": cfg["dataset"],
            "seed": cfg["seed"],
            "model_summary": r["model_summary"],
            "optimizer": cfg["optimizer"],
            "lr": cfg["lr"],
            "momentum": cfg["momentum"],
            "weight_decay": cfg["weight_decay"],
            "epochs_trained": r["epochs_trained"],
            "best_val_accuracy": r["best_val_accuracy"],
            "best_val_loss": r["best_val_loss"],
        }
    )

df = pd.DataFrame(rows)
df.to_csv(artifacts_dir / "runs.csv", index=False)
df

torch.save(best_overall["best_state_dict"], artifacts_dir / "best_model.pt")


# Блок 10. Графики
hist = best_overall["history"]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(hist["train_loss"], label="train_loss")
plt.plot(hist["val_loss"], label="val_loss")
plt.title("E4: loss")
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(hist["train_acc"], label="train_acc")
plt.plot(hist["val_acc"], label="val_acc")
plt.title("E4: accuracy")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig(fig_dir / "curves_best.png")
plt.close()

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(O1["history"]["val_loss"], label="O1 (lr=1e-1)")
plt.plot(O2["history"]["val_loss"], label="O2 (lr=1e-5)")
plt.title("LR extremes: val_loss")
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(O1["history"]["val_acc"], label="O1 (lr=1e-1)")
plt.plot(O2["history"]["val_acc"], label="O2 (lr=1e-5)")
plt.title("LR extremes: val_acc")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig(fig_dir / "curves_lr_extremes.png")
plt.close()


# Блок 11. Финальная оценка лучшей модели на test (один раз)
best_model = MLP(
    hidden_sizes=best_cfg["hidden_sizes"],
    dropout_p=best_cfg["dropout_p"],
    use_batchnorm=best_cfg["use_batchnorm"],
).to(device)
best_model.load_state_dict(best_overall["best_state_dict"])

criterion = nn.CrossEntropyLoss()
test_loss, test_acc = evaluate(best_model, test_loader, criterion, device)
print("\nТочность на test:", test_acc)

best_config = dict(best_overall["config"])
best_config.update(
    {
        "epochs_trained": best_overall["epochs_trained"],
        "best_val_accuracy": best_overall["best_val_accuracy"],
        "best_val_loss": best_overall["best_val_loss"],
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
    }
)

with open(artifacts_dir / "best_config.json", "w", encoding="utf-8") as f:
    json.dump(best_config, f, ensure_ascii=False, indent=2)

# Короткое резюме (как итоговый вывод)
summary = {
    "device": str(device),
    "input_dim": int(input_dim),
    "num_classes": int(num_classes),
    "picked_from": str(best_reg["experiment_id"]),
    "best_val_accuracy": float(best_overall["best_val_accuracy"]),
    "best_val_loss": float(best_overall["best_val_loss"]),
    "test_accuracy": float(test_acc),
    "test_loss": float(test_loss),
    "runs_csv": str(artifacts_dir / "runs.csv"),
    "best_model_path": str(artifacts_dir / "best_model.pt"),
    "best_config_path": str(artifacts_dir / "best_config.json"),
    "fig_best": str(fig_dir / "curves_best.png"),
    "fig_lr": str(fig_dir / "curves_lr_extremes.png"),
    "fast_mode": bool(CFG["data"]["fast_mode"]),
}
summary


Device: cpu
Batch: torch.Size([128, 3, 32, 32]) torch.Size([128])
Dataset: CIFAR10 | input_dim: 3072 | num_classes: 10

Часть A (S08): регуляризация
[E1] Итог: best_val_acc=0.4604 | epochs=20
[E2] Итог: best_val_acc=0.4779 | epochs=20
[E3] Итог: best_val_acc=0.4746 | epochs=20

Лучшая регуляризация: E2 val_acc= 0.47791666666666666
[E4] EarlyStopping: остановка (patience=4)
[E4] Итог: best_val_acc=0.4717 | epochs=14

Часть B (S09): LR, оптимизаторы, weight decay
[O1] Итог: best_val_acc=0.1100 | epochs=6
[O2] Итог: best_val_acc=0.3250 | epochs=6
[O3] Итог: best_val_acc=0.4746 | epochs=15

Точность на test: 0.4652


{'device': 'cpu',
 'input_dim': 3072,
 'num_classes': 10,
 'picked_from': 'E2',
 'best_val_accuracy': 0.47166666626930237,
 'best_val_loss': 1.5136073112487793,
 'test_accuracy': 0.4652,
 'test_loss': 1.5230304586410524,
 'runs_csv': 'c:\\Users\\L\\gribishok\\makaroni_po_flotski\\homeworks\\HW08-09\\artifacts\\runs.csv',
 'best_model_path': 'c:\\Users\\L\\gribishok\\makaroni_po_flotski\\homeworks\\HW08-09\\artifacts\\best_model.pt',
 'best_config_path': 'c:\\Users\\L\\gribishok\\makaroni_po_flotski\\homeworks\\HW08-09\\artifacts\\best_config.json',
 'fig_best': 'c:\\Users\\L\\gribishok\\makaroni_po_flotski\\homeworks\\HW08-09\\artifacts\\figures\\curves_best.png',
 'fig_lr': 'c:\\Users\\L\\gribishok\\makaroni_po_flotski\\homeworks\\HW08-09\\artifacts\\figures\\curves_lr_extremes.png',
 'fast_mode': True}